In [3]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# ── Read files ────────────────────────────────────────────────────────────────
roads   = pd.read_csv("_roads3.csv")
bridges = pd.read_excel("BMMS_overview.xlsx")

# ── Haversine distance (km) ───────────────────────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# ── Nearest point within threshold ───────────────────────────────────────────
def nearest(lat, lon, df, threshold):
    dists = np.array([haversine(lat, lon, r, c)
                      for r, c in zip(df["lat"], df["lon"])])
    i = dists.argmin()
    return (i, dists[i]) if dists[i] <= threshold else (None, None)

# ── Road length in km ────────────────────────────────────────────────────────
def road_length(df):
    coords = df[["lat","lon"]].values
    return sum(haversine(coords[i,0], coords[i,1],
                         coords[i+1,0], coords[i+1,1])
               for i in range(len(coords)-1))

# ── Does a road intersect N1 or N2? ──────────────────────────────────────────
def intersects_main(group, main_df, threshold_km=0.5):
    for _, row in group.iterrows():
        i, dist = nearest(row["lat"], row["lon"], main_df, threshold_km)
        if i is not None:
            return True
    return False

# ── Select N1 and N2 ─────────────────────────────────────────────────────────
n1 = roads[roads["road"] == "N1"].reset_index(drop=True)
n2 = roads[roads["road"] == "N2"].reset_index(drop=True)

# ── Select side roads ────────────────────────────────────────────────────────
n_roads = roads[roads["road"].str.match(r"^N\d+$") &
                ~roads["road"].isin(["N1", "N2"])].copy()

side_roads = []
for road_name, group in n_roads.groupby("road"):
    group = group.reset_index(drop=True)
    if road_length(group) > 25 and \
       (intersects_main(group, n1) or intersects_main(group, n2)):
        side_roads.append(group)

side = pd.concat(side_roads, ignore_index=True) if side_roads else \
       pd.DataFrame(columns=roads.columns)

print(f"Side roads selected: {side['road'].unique().tolist() if not side.empty else []}")

# ── Preprocess a single road ──────────────────────────────────────────────────
def preprocess(df, road_name):
    df = df[["road","lrp","lat","lon","chainage"]].copy()

    b = bridges[bridges["road"] == road_name][
            ["LRPName","length","condition","constructionYear"]]
    df = df.merge(b, how="left", left_on="lrp", right_on="LRPName")

    df["model_type"] = np.select([df["length"].notna()], ["bridge"], default="link")

    df.iloc[0,  df.columns.get_loc("model_type")] = "sourcesink"
    df.iloc[-1, df.columns.get_loc("model_type")] = "sourcesink"

    df["length"] = df["length"].fillna(
        (df["chainage"].shift(-1) - df["chainage"]) * 1000)

    df["constructionYear"] = pd.to_numeric(df["constructionYear"], errors="coerce")
    df["quality_score"]    = df["condition"].map(
        {"A":4,"B":3,"C":2,"D":1}).fillna(0)

    df = df.rename(columns={"lrp": "name"})
    df = df.drop_duplicates("name", keep="last")
    return df.drop(columns=["LRPName","chainage"], errors="ignore").reset_index(drop=True)

n1_p   = preprocess(n1, "N1")
n2_p   = preprocess(n2, "N2")
side_p = pd.concat([preprocess(g.reset_index(drop=True), r)
                    for r, g in side.groupby("road")],
                   ignore_index=True) if not side.empty else pd.DataFrame()

# ── Assemble road rows and assign stable IDs ──────────────────────────────────
COLS = ["road","model_type","condition","name","lat","lon","length"]

road_frames = [n1_p, n2_p]
if not side_p.empty:
    road_frames.append(side_p)

roads_out = pd.concat(road_frames, ignore_index=True)
for c in COLS:
    if c not in roads_out.columns:
        roads_out[c] = np.nan
roads_out = roads_out[COLS].copy()
roads_out.insert(1, "id", range(1, len(roads_out) + 1))

# ── Mark intersections IN-PLACE + append one extra row for the connecting road
# ─────────────────────────────────────────────────────────────────────────────
# Key insight: the intersection node must stay in its correct sequential
# position within road_a so that generate_model builds graph edges correctly.
#
# Strategy:
#   1. Change the matched road_a node's model_type to "intersection" IN-PLACE
#      in roads_out — it stays exactly where it is in the road sequence.
#   2. Append ONE new row for road_b with the same canonical ID — when
#      generate_model later processes road_b's rows, it will encounter this
#      ID and add a graph edge from road_b's previous node to the intersection,
#      stitching the two roads together.
#
# Result: each intersection ID appears exactly twice — once in-sequence on
# road_a, once appended for road_b. No duplicates, no ordering problems.

extra_rows = []    # road_b rows to append at the end
claimed_ids = set()  # global: each node becomes an intersection at most once


def find_intersections(df_a, df_b, road_a, road_b, threshold_km=0.5):
    """
    Snap each endpoint/node of df_b to df_a within threshold_km.
    Mutates roads_out in-place for road_a nodes; appends to extra_rows for road_b.
    """
    matched_a_indices = set()

    # Pre-compute the roads_out slice for road_a with its original index preserved
    road_a_slice = roads_out[roads_out["road"] == road_a].reset_index()
    # road_a_slice columns: index (=roads_out index), road, id, model_type, ...

    for _, rb in df_b.iterrows():
        i, dist = nearest(rb.lat, rb.lon, df_a, threshold_km)
        if i is None or i >= len(road_a_slice):
            continue

        global_idx = road_a_slice.iloc[i]["index"]   # index into roads_out
        canon_id   = int(roads_out.at[global_idx, "id"])

        # Skip if already claimed globally or already matched in this call
        if canon_id in claimed_ids or i in matched_a_indices:
            continue

        matched_a_indices.add(i)
        claimed_ids.add(canon_id)

        # 1. Mark the road_a node as intersection in-place (keeps sequence intact)
        roads_out.at[global_idx, "model_type"] = "intersection"
        roads_out.at[global_idx, "length"]     = 0.0

        # 2. Append a road_b row with the same canonical ID
        extra_rows.append({
            "road":       road_b,
            "id":         canon_id,
            "model_type": "intersection",
            "condition":  np.nan,
            "name":       np.nan,
            "lat":        round(float(rb.lat), 6),
            "lon":        round(float(rb.lon), 6),
            "length":     0.0,
        })


# N1 ↔ N2
find_intersections(n1_p, n2_p, "N1", "N2")

# Side roads ↔ N1 and N2
if not side_p.empty:
    for side_road_name, side_road_df in side_p.groupby("road"):
        srdf = side_road_df.reset_index(drop=True)
        find_intersections(n1_p, srdf, "N1", side_road_name)
        find_intersections(n2_p, srdf, "N2", side_road_name)

# ── Final assembly ────────────────────────────────────────────────────────────
out_cols = ["road","id","model_type","condition","name","lat","lon","length"]

if extra_rows:
    extra_df = pd.DataFrame(extra_rows)[out_cols]
    out = pd.concat([roads_out[out_cols], extra_df], ignore_index=True)
else:
    out = roads_out[out_cols].copy()

out[["lat","lon","length"]] = out[["lat","lon","length"]].round(6)

# Clear name for non-sourcesink types
edge_types = {"source", "sink", "sourcesink"}
out.loc[~out["model_type"].isin(edge_types), "name"] = np.nan

out.to_csv("processed_data.csv", index=False)
print(f"\nDone: {len(out)} rows")
print(out["model_type"].value_counts().to_string())

# ── Verification ──────────────────────────────────────────────────────────────
print("\nIntersection rows (same ID should appear on exactly two different roads):")
int_rows = out[out["model_type"] == "intersection"][["road","id","lat","lon"]]
print(int_rows.sort_values("id").to_string(index=False))

print("\nID sharing check:")
all_ok = True
for iid in int_rows["id"].unique():
    roads_sharing = out[out["id"] == iid]["road"].tolist()
    if len(roads_sharing) != 2:
        status = f"WARNING: {len(roads_sharing)} rows"
        all_ok = False
    else:
        status = "OK"
    print(f"  ID {iid}: {roads_sharing}  [{status}]")

if all_ok:
    print("\nAll intersection IDs appear exactly twice. ✓")
else:
    print("\nSome IDs have unexpected row counts — check warnings above.")

Side roads selected: ['N102', 'N104', 'N105', 'N204', 'N207', 'N208']

Done: 3134 rows
model_type
link            2025
bridge          1065
intersection      28
sourcesink        16

Intersection rows (same ID should appear on exactly two different roads):
road   id       lat       lon
  N1   18 23.706083 90.521527
  N2   18 23.705917 90.521444
  N1   29 23.690416 90.546583
N105   29 23.690416 90.546611
  N1  184 23.478944 91.117722
N102  184 23.481583 91.116777
  N1  186 23.478972 91.118166
N102  186 23.478972 91.118194
  N1  336 23.009556 91.381360
N104  336 23.009528 91.381444
  N2 1359 23.785389 90.568888
N105 1359 23.785194 90.568805
  N2 1568 24.050833 91.114444
N102 1568 24.050611 91.114667
N204 1656 24.147916 91.346611
  N2 1656 24.147861 91.346444
  N2 1658 24.149889 91.347444
N204 1658 24.148361 91.351389
N204 1749 24.266556 91.477860
  N2 1749 24.267694 91.476888
N207 1762 24.294861 91.510250
  N2 1762 24.294722 91.510083
N207 1764 24.296416 91.512083
  N2 1764 24.297889 91.